# 00 Prepare Datasets

This notebook creates the v2 final dataset layer used by the paper-facing experiments. Alpha stays full-range for training/testing, Beta is filtered to the one-year study period, and Gamma is selected as a one-site case study from Beta.


## 1. Imports And Paths

Resolve the article root, load `journal_v2` config, and show the output locations before writing anything.


In [ ]:
from pathlib import Path
import sys

# Keep notebook imports stable whether the notebook is run from JupyterLab,
# VS Code, or the repository root.
article_root = Path.cwd()
while article_root.name != "2_journal_article":
    if article_root.parent == article_root:
        raise RuntimeError("Could not locate publication/2_journal_article")
    article_root = article_root.parent
notebook_dir = article_root / "notebooks"
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

import _experiment_helpers as h

cfg = h.load_config(article_root)
paths = h.article_paths(article_root, cfg)
h.ensure_output_dirs(paths)
print(f"Article root: {article_root}")
print(f"Config schema: {cfg['schema_version']}")
print(f"Output root: {paths.outputs}")

resolved_paths = {
    "alpha_final": article_root / cfg["paths"]["alpha_dataset_path"],
    "beta_final": article_root / cfg["paths"]["beta_dataset_path"],
    "gamma_final": article_root / cfg["paths"]["gamma_dataset_path"],
    "dataset_final_dir": paths.final,
}
resolved_paths


## 2. Optional Gamma Site Override

Review the Gamma candidate table first when possible. Leave this as `None` to use the config-driven automatic selection, or set it to a site such as `"beta_B"` for an explicit case-study choice.


In [ ]:
# Optional override after reviewing Gamma candidates.
# Set to None to use config auto-selection.
GAMMA_SITE_OVERRIDE = None  # e.g. "beta_B"


## 3. Build Final Datasets

This writes Alpha, Beta, and Gamma Parquet files plus validation summaries. It preserves the original seven-column dataset schema in the final Parquet payloads.


In [ ]:
result = h.run_prepare_datasets(article_root, gamma_site_override=GAMMA_SITE_OVERRIDE)
final_summary = result["final_summary"]
final_summary


## 4. Review Site Rankings

Alpha rankings define the top leave-one-station-out folds. Beta rankings show why the selected Gamma site is a strong candidate for the forecast-impact case study.


In [ ]:
alpha_rank = h.site_rpf_summary(result["alpha"], "Alpha").sort_values(
    ["rpf_days", "rpf_intervals", "substation_id"],
    ascending=[False, False, True],
)
gamma_rank = result["gamma_rankings"]
print(f"Selected Gamma site: {result['gamma_site']}")
display(alpha_rank.head(10))
display(gamma_rank.head(10))


## 5. Validation Summary

These checks confirm the v2 final dataset assumptions: one-year Beta, one-site Gamma, and expected provisional row counts.


In [ ]:
validation = result["validation"]
validation
